# 🛍️ Project 03: Customer Churn & Sales Trend Analysis
**Organization / Internship:** Intern Circle  
**Track:** Data Science & Machine Learning  
**Project Goal:** Perform end-to-end business intelligence and analytical modeling on an e-commerce transactions dataset to evaluate revenue trends, compute MoM growth, execute **RFM (Recency, Frequency, Monetary) Customer Segmentation**, identify churn, and formulate actionable growth recommendations.

---

### 📋 Project Requirements Checklist
- [x] **Feature Engineering & Date Extraction:** Converting timestamps, extracting Year, Month, Day, DayOfWeek, Quarter, Hour, Weekend indicators
- [x] **Pivot Tables & Trend Calculations:** Monthly revenue trajectory, MoM revenue percentage growth, multi-dimensional category and regional cross-tabs
- [x] **RFM Customer Segmentation & Churn Detection:** Calculating Recency, Frequency, Monetary quantiles; identifying Champions, Loyalists, At Risk, and Lost segments
- [x] **Visual Analytics:** Dual-axis revenue/order charts, segment distribution, and hourly sales intensity heatmaps
- [x] **Actionable Business Recommendations Summary:** High-impact commercial strategies to increase retention and lifetime value


## 1. Setup & Library Imports
Importing essential data analysis and visualization toolkits.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 11

print("Environment successfully initialized!")


## 2. Dataset Ingestion & Date Feature Engineering
Loading transaction records and extracting calendar and temporal features to analyze cyclic buying patterns.


In [ ]:
# Load transactions
df = pd.read_csv("data/ecommerce_sales_data.csv")

# Date extraction
df["Order_Date"] = pd.to_datetime(df["Order_Date"])
df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.strftime("%b")
df["Year_Month"] = df["Order_Date"].dt.to_period("M").astype(str)
df["Quarter"] = "Q" + df["Order_Date"].dt.quarter.astype(str)
df["Day_of_Week"] = df["Order_Date"].dt.day_of_week
df["Day_Name"] = df["Order_Date"].dt.day_name()
df["Hour"] = df["Order_Date"].dt.hour
df["Is_Weekend"] = df["Day_of_Week"].isin([5, 6]).astype(int)

print(f"Total Transactions: {df.shape[0]} | Columns: {df.shape[1]}")
df.head()


## 3. Revenue Trends & Pivot Table Calculations
Evaluating month-over-month (MoM) revenue trends, Average Order Value (AOV), and category distributions.


In [ ]:
# Monthly Revenue Trend & MoM Growth
monthly_trend = df.groupby("Year_Month").agg(
    Total_Revenue=("Total_Revenue", "sum"),
    Total_Orders=("Order_ID", "count"),
    Average_Order_Value=("Total_Revenue", "mean"),
    Unique_Customers=("Customer_ID", "nunique")
).reset_index()

monthly_trend["MoM_Growth_Pct"] = monthly_trend["Total_Revenue"].pct_change() * 100
monthly_trend.round(2)


In [ ]:
# Pivot Table: Category Revenue across Quarters ($)
pivot_category_quarter = df.pivot_table(
    index="Category",
    columns="Quarter",
    values="Total_Revenue",
    aggfunc="sum",
    margins=True,
    margins_name="Total"
).round(2)
pivot_category_quarter


In [ ]:
# Pivot Table: Regional Revenue by Payment Method ($)
pivot_region_payment = df.pivot_table(
    index="Region",
    columns="Payment_Method",
    values="Total_Revenue",
    aggfunc="sum",
    margins=True,
    margins_name="Total"
).round(2)
pivot_region_payment


## 4. RFM (Recency, Frequency, Monetary) Segmentation & Churn
Segmenting customers by loyalty and revenue contribution to spot customer churn patterns.


In [ ]:
# Calculate RFM metrics
snapshot_date = df["Order_Date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("Customer_ID").agg(
    Recency_Days=("Order_Date", lambda x: (snapshot_date - x.max()).days),
    Frequency=("Order_ID", "count"),
    Monetary=("Total_Revenue", "sum")
).reset_index()

# Quantile scores (1 to 4)
rfm["R_Score"] = pd.qcut(rfm["Recency_Days"], q=4, labels=[4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), q=4, labels=[1, 2, 3, 4]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"), q=4, labels=[1, 2, 3, 4]).astype(int)

# Segment assignment
def assign_segment(row):
    r, f, m = row["R_Score"], row["F_Score"], row["M_Score"]
    if r >= 3 and f >= 3 and m >= 3:
        return "Champions"
    elif r >= 3 and f >= 2:
        return "Loyal Customers"
    elif r >= 3 and f == 1:
        return "Promising New"
    elif r == 2 and f >= 2:
        return "Needs Attention"
    elif r == 1 and f >= 2:
        return "At Risk"
    else:
        return "Lost / Churned"

rfm["Customer_Segment"] = rfm.apply(assign_segment, axis=1)
rfm["Is_Churned"] = (rfm["Recency_Days"] >= 90).astype(int)

rfm.head(10)


In [ ]:
# Summary across Customer Segments
segment_summary = rfm.groupby("Customer_Segment").agg(
    Customer_Count=("Customer_ID", "count"),
    Avg_Recency=("Recency_Days", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Total_Revenue=("Monetary", "sum"),
    Avg_Revenue_Per_Cust=("Monetary", "mean"),
    Churn_Rate=("Is_Churned", "mean")
).round(2).reset_index()

segment_summary["Customer_Pct"] = (segment_summary["Customer_Count"] / len(rfm) * 100).round(2)
segment_summary["Revenue_Pct"] = (segment_summary["Total_Revenue"] / rfm["Monetary"].sum() * 100).round(2)
segment_summary


## 5. Visual Analytics & Heatmaps
Visualizing time series trends, customer segment revenue distributions, and peak checkout times.


In [ ]:
# 1. Dual-Axis Monthly Trend: Revenue vs Order Volume
monthly_data = df.groupby("Year_Month").agg(
    Revenue=("Total_Revenue", "sum"),
    Orders=("Order_ID", "count")
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

p1 = ax1.plot(monthly_data["Year_Month"], monthly_data["Revenue"] / 1000, color="#2b5c8f", marker="o", linewidth=2.5, label="Revenue ($K)")
p2 = ax2.plot(monthly_data["Year_Month"], monthly_data["Orders"], color="#d95f02", marker="s", linestyle="--", linewidth=2, label="Order Count")

ax1.set_title("E-Commerce Monthly Revenue & Order Volume Trends", fontsize=14, fontweight="bold", pad=12)
ax1.set_xlabel("Year-Month", fontsize=11)
ax1.set_ylabel("Revenue ($ in Thousands)", color="#2b5c8f", fontsize=11)
ax2.set_ylabel("Order Count", color="#d95f02", fontsize=11)
ax1.tick_params(axis='x', rotation=45)

lines = p1 + p2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper left", frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
# 2. RFM Customer Segments: Customer Count vs. Revenue Contribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=segment_summary, x="Customer_Count", y="Customer_Segment", hue="Customer_Segment", palette="Blues_r", legend=False, ax=axes[0])
axes[0].set_title("Customer Count by Segment")
axes[0].set_xlabel("Customer Count")

sns.barplot(data=segment_summary, x="Revenue_Pct", y="Customer_Segment", hue="Customer_Segment", palette="Greens_r", legend=False, ax=axes[1])
axes[1].set_title("Share of Total Revenue (%)")
axes[1].set_xlabel("Revenue Share (%)")

plt.tight_layout()
plt.show()


In [ ]:
# 3. Revenue Intensity Heatmap: Day of Week vs. Hour of Day
days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heatmap_matrix = df.pivot_table(
    index="Day_Name",
    columns="Hour",
    values="Total_Revenue",
    aggfunc="sum",
    fill_value=0
).reindex(days_order)

plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_matrix / 1000, cmap="YlGnBu", annot=False, cbar_kws={'label': 'Revenue ($ Thousands)'})
plt.title("Revenue Concentration Heatmap: Day of Week vs. Hour of Day", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Hour of Day (0 to 23)")
plt.ylabel("Day of Week")
plt.tight_layout()
plt.show()


## 6. Actionable Business Recommendations
1. **VIP Loyalty & Rewards for 'Champions':**
   - Provide premium perks (free express delivery, concierge support) to protect the top revenue segment.
2. **Win-Back Automated Email Campaigns for 'At Risk' Accounts:**
   - Detect inactivity at day 60 and trigger exclusive limited-time 15% discount incentives.
3. **Optimized Marketing Timing:**
   - Deploy promotional pushes and email broadcasts between 14:00 and 19:00, aligned with peak transactional velocity.
4. **Checkout Cross-Selling:**
   - Recommend high-margin accessories during checkout to boost Average Order Value (AOV).
